# Embedding 微调：数据、负例、训练、评估与导出

## Goal
运行一套模型和硬件可配置的 embedding 微调流程。默认使用离线随机 tiny BERT 和合成检索数据，验证真实训练代码能完整运行。**示例指标不能代表业务效果**。

产物：训练负例、验证集选择记录、基线/最终检索指标、可用原生 SentenceTransformer 加载的模型。

## Setup
从仓库根目录安装：`python -m pip install -e '.[embedding,notebooks]'`。
Colab 可使用 `%pip install -e '/content/advai-notebooks[embedding,notebooks]'`；先把仓库放到该路径，再修改下一单元的 ROOT。

In [ ]:
from pathlib import Path
import sys
import importlib.metadata

# Colab: set ROOT = Path("/content/advai-notebooks") after placing the repo there.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Set ROOT to the repository directory before continuing")
sys.path.insert(0, str(ROOT / "src"))
for name in ["torch", "transformers", "sentence-transformers"]:
    print(name, importlib.metadata.version(name))
print("Repository:", ROOT)
from huggingface_hub.utils import disable_progress_bars
from transformers.utils.logging import disable_progress_bar
disable_progress_bars()
disable_progress_bar()


## Step 1 — 配置和数据检查
参数来自 YAML。可配置基座、版本、query/document 前缀、pooling、长度、loss、批次、梯度累积、设备和精度。每次 notebook 执行创建新实验目录，避免覆盖历史产物。

In [ ]:
import tempfile
from advai_notebooks.embedding.config import load_config
from advai_notebooks.common.retrieval_data import load_data

CONFIG = ROOT / "configs/embedding/smoke.yaml"  # Domain data: configs/embedding/bge.yaml
cfg = load_config(CONFIG, root=ROOT)
parent = ROOT / "outputs/embedding"
parent.mkdir(parents=True, exist_ok=True)
RUN_DIR = Path(tempfile.mkdtemp(prefix="notebook-", dir=parent)) / "run"
cfg.output_dir = str(RUN_DIR)
data = load_data(cfg)
print(data.summary())
print("Model:", cfg.model)
print("Runtime:", cfg.runtime)
print("Run:", RUN_DIR)

## Step 2 — 检查独立准备的训练样本
训练只读取 `data.training_samples`，不负责负例挖掘。数据准备可单独运行 `python -m advai_notebooks.data_preparation prepare --config configs/data_preparation/smoke.yaml`，再将生成的 embedding_samples.json 路径写入训练配置。

默认示例已经有已判断正负例。查询、正负文档 ID 会与 qrels 校验；验证/测试查询和已知正例不能充当训练负例。采样仍避开已知批内假负例。

In [ ]:
from advai_notebooks.embedding.data import load_training_samples
from advai_notebooks.embedding.sampling import embedding_batches

rows, samples_fingerprint = load_training_samples(cfg, data)
batches = embedding_batches(rows, data, cfg.training.batch_size, cfg.training.seed)
print("Prepared rows:", len(rows), "Sample fingerprint:", samples_fingerprint)
print("First batch sizes:", [len(b) for b in batches[:10]])
for row in rows[:2]:
    print(row)
    print("Query:", data.queries[row["query_id"]]["text"])
    print("Positive:", data.corpus[row["positive_id"]]["text"])
    print("Negatives:", [data.corpus[d]["text"] for d in row["negative_ids"]])

## Step 3 — 训练、验证集选择、测试集报告
MNRL 学习区分配对正例与批内/显式困难负例。可切换 `cached_mnrl` 降低训练激活显存。梯度累积不会扩大单个 loss 的负例池。

验证指标选择 best，包括训练前的 epoch 0；测试集不用于选择。若训练未改善验证指标，导出基线并明确记录 `best_epoch=0`。

In [ ]:
from advai_notebooks.embedding.pipeline import run
report = run(cfg)
print("Selected epoch:", report["training"]["best_epoch"])
print("Export:", report["export"])

## Checks — 训练轨迹与检索指标
Recall 以全部已知正例为分母；MRR 看首个相关结果；nDCG 使用分级 relevance。未标注文档在离线指标中记为零增益。搜索全语料，不把遗漏正例补进候选。

In [ ]:
import math
from IPython.display import display, HTML
import matplotlib.pyplot as plt

history = report["training"]["history"]
assert all(math.isfinite(h["train_loss"]) for h in history[1:])
K = 3 if 3 in cfg.evaluation.cutoffs else (10 if 10 in cfg.evaluation.cutoffs else cfg.evaluation.cutoffs[0])
keys = [f"{metric}@{K}" for metric in ["recall", "mrr", "ndcg"]]
rows = [(name, report[field]["metrics"]) for name, field in
        [("Before training", "baseline_test"), ("Selected export", "finetuned_test")]]
html = "<table><tr><th>Checkpoint</th>" + "".join(f"<th>{k}</th>" for k in keys) + "</tr>"
for name, metrics in rows:
    html += f"<tr><td>{name}</td>" + "".join(f"<td>{metrics[k]:.4f}</td>" for k in keys) + "</tr>"
display(HTML(html + "</table>"))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot([h["epoch"] for h in history[1:]], [h["train_loss"] for h in history[1:]], marker="o")
axes[0].set(xlabel="Epoch", ylabel="Contrastive loss", title="Training loss")
selection_metric = cfg.evaluation.selection_metric
axes[1].plot([h["epoch"] for h in history], [h["validation"][selection_metric] for h in history], marker="o")
axes[1].set(xlabel="Epoch (0 = baseline)", ylabel=selection_metric, title="Validation selection")
axes[1].set_ylim(0, 1.05)
fig.suptitle("Tiny synthetic smoke run; quality is not a domain benchmark")
fig.tight_layout()
fig.savefig(RUN_DIR / "training.png", dpi=130)
plt.show()

## Step 4 — 重载导出并验证原生 SDK 一致性
Query 和 document 前缀会保存为原生 prompts。SDK 使用 `encode_query` / `encode_document`；wrapper 的 `encode` 明确指定 role。输出已归一化，可使用点积计算 cosine 相似度。

In [ ]:
from advai_notebooks.embedding.model import EmbeddingModel
import numpy as np
from sentence_transformers import SentenceTransformer

exported = EmbeddingModel.load(report["export"], device="cpu")
sdk = SentenceTransformer(report["export"], device="cpu")
texts = ["reset password", "card charged twice"]
vectors = exported.encode(texts, role="query").numpy()
np.testing.assert_allclose(vectors, sdk.encode_query(texts, show_progress_bar=False), atol=1e-6)
np.testing.assert_allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-6)
print("Reload and native SDK agreement passed. Shape:", vectors.shape)
print("Artifacts:", sorted(p.name for p in RUN_DIR.iterdir()))

## Next steps
- 使用独立来源的业务查询与语料，按 `data/sample/retrieval/README.md` 准备 corpus、queries、qrels，补充明确负例。
- 切换 `configs/embedding/bge.yaml` 或兼容的单向量 encoder，按模型要求填写前缀与 pooling，固定 revision。
- 按显存调整 batch、cached mini batch、max_len 与 gradient checkpointing。CUDA 可选 fp16/bf16；CPU/MPS 使用 fp32。
- Ranker 可独立读取自己的训练样本；组合检索在 `notebooks/workflows/01_retrieval_comparison.ipynb` 中执行。

v1 为单设备全参数训练；保存的是推理 checkpoint，尚未实现 embedding/ranker 的优化器状态恢复。用 latest 开始新实验时会新建优化器。